# VoteLens: document-grounded claim verification
## HackApertus Track 2A — executable demonstration

VoteLens classifies a claim as **entailment** (0), **neutral** (1), or **contradiction** (2) against supplied Swiss voting information. It supports German, French and Italian, including cross-language source–claim pairs.

This notebook demonstrates the current **`vote-aware-v1`** application using **CSCS Apertus v1.5 70B**. It invokes the repository-root `make run`; it does not reimplement the classifier.

**Execution status:** both current-version Docker runs completed successfully on 15 October 2026 using CSCS Apertus v1.5 70B. The saved outputs below record Task B (519 input tokens, 71 output tokens, 1,096 ms) and Task A (1,345 input tokens, 93 output tokens, 1,524 ms). Both exited zero and returned entailment; Task A cited physical PDF page 19. These are two smoke examples, not an accuracy benchmark. Live execution is disabled by default for viewing.


## Reproduction

1. Place this notebook at the VoteLens repository root, alongside the root Makefile. Start JupyterLab there and select a Python 3 kernel. Alternatively, set `VOTELENS_PROJECT_ROOT` to the checkout directory before starting Jupyter.
2. Docker with Compose must be available and running; GNU Make must be installed. The notebook kernel requires IPython. Application dependencies are built inside Docker.
3. Supply a valid inference credential through `LLM_API_KEY` or `CSCS_INFERENCE_API_KEY`. The optional hidden prompt below is available when a credential was not inherited. Never write a credential in a notebook cell.
4. Set `LIVE_RUN=True` in the configuration cell to enable execution, then run the notebook in order. Two completion calls are planned, with endpoint checks; they consume API resources.

Task A needs access to the PDF and SentenceSwissBERT weights, or their existing caches. First-time downloads and Docker builds can take longer than inference. The encoder runs on CPU; the large language model runs remotely. No local NVIDIA GPU is required for this external-endpoint demonstration.


In [8]:
from pathlib import Path
import json, os, subprocess, uuid, hashlib, shutil
from html import escape
from IPython.display import display, HTML

# Place this notebook at the repository root, or set VOTELENS_PROJECT_ROOT.
def find_project():
    configured = os.environ.get("VOTELENS_PROJECT_ROOT")
    candidates = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "track_2a" / "compose.submission.yaml").is_file():
            return candidate.resolve()
    raise RuntimeError("Place this notebook in the VoteLens repository and start Jupyter there, or set VOTELENS_PROJECT_ROOT.")

PROJECT = find_project()
LIVE_RUN = False  # Opt-in: real inference consumes endpoint resources.
MODEL = "swiss-ai/Apertus-v1.5-70B"
BASE_URL = "https://api.inference.cscs.ch/v1"
CLAIM = "Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité."

REQUEST_B = {
    "id": "notebook-task-B",
    "reference": {"text": CLAIM, "language": "fr"},
    "claim": {"text": CLAIM, "language": "fr"},
}
REQUEST_A = {
    "id": "notebook-task-A",
    "booklet": {
        "path": "https://www.bk.admin.ch/dam/fr/sd-web/WeUrKyC0FyPc/2026-06-14_explications_du_conseil_federal.pdf",
        "language": "fr",
    },
    "vote": "Initiative populaire « Pas de Suisse à 10 millions ! (initiative pour la durabilité) »",
    "claim": {"text": CLAIM, "language": "fr"},
}

def show_result(result, origin):
    display(HTML(f"<h3>{escape(origin)}: {escape(result['label_name'])}</h3>"))
    for item in result["evidence"]:
        page = "Supplied reference" if item["page"] is None else f"PDF page {item['page']}"
        display(HTML(f"<p><b>{escape(page)}</b></p><blockquote>{escape(item['text'])}</blockquote>"))
    rows = "".join(f"<tr><td>{escape(k)}</td><td>{v}</td></tr>" for k,v in result["metrics"].items())
    display(HTML(f"<table><tr><th>Metric</th><th>Value</th></tr>{rows}</table>"))

SOURCE_HASHES = {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest()
                 for name in ["track_2a/src/booklet_app.py", "track_2a/compose.submission.yaml"]}
print("Pipeline: vote-aware-v1; endpoint model:", MODEL)
print("Source fingerprints:", json.dumps(SOURCE_HASHES, indent=2))


Pipeline: vote-aware-v1; endpoint model: swiss-ai/Apertus-v1.5-70B
Source fingerprints: {
  "track_2a/src/booklet_app.py": "da6f5236937a9aecee9611266f4735bb0e4c23cc1e60fd6356b5870a5b80c1d2",
  "track_2a/compose.submission.yaml": "3a833d206c4602e30eee6fd9b47bd33ab516c187b11d3c8e5d8b02be27e18591"
}


## Task B — supplied reference

The source and claim both state that the Federal Council recommends rejecting the sustainability initiative. The expected label is **entailment**. This example isolates source-based judgement from PDF extraction and retrieval. A reference has no physical PDF page, so evidence uses `page: null`.


In [9]:
print(json.dumps(REQUEST_B, ensure_ascii=False, indent=2))


{
  "id": "notebook-task-B",
  "reference": {
    "text": "Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.",
    "language": "fr"
  },
  "claim": {
    "text": "Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.",
    "language": "fr"
  }
}


## Task A — voting booklet

The request supplies the full French Federal Chancellery booklet, the proposal name and the same claim. The expected label is **entailment**.

The application extracts physical PDF pages with Poppler, constructs passages and retrieves the top five using SentenceSwissBERT. In `vote-aware-v1`, retrieval uses both the proposal name and claim. Apertus must resolve the specified proposal and choose source passage IDs for evidence. Unknown or ambiguous proposals fail explicitly. The final output includes source text and physical page numbers.

This demonstration uses French-to-French inputs; it does not test the full multilingual matrix.


In [10]:
print(json.dumps(REQUEST_A, ensure_ascii=False, indent=2))


{
  "id": "notebook-task-A",
  "booklet": {
    "path": "https://www.bk.admin.ch/dam/fr/sd-web/WeUrKyC0FyPc/2026-06-14_explications_du_conseil_federal.pdf",
    "language": "fr"
  },
  "vote": "Initiative populaire « Pas de Suisse à 10 millions ! (initiative pour la durabilité) »",
  "claim": {
    "text": "Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.",
    "language": "fr"
  }
}


## Credential and Docker execution

Credentials are supplied privately and are never printed. Each run writes uniquely named input, result and diagnostic files under `track_2a/data/notebook-demo/`, leaving the committed `request.json` unchanged. Full diagnostics are not displayed in this notebook. A failed run raises an error; it is never replaced by a stored success or a guessed label.


In [11]:
if LIVE_RUN and not (os.environ.get("LLM_API_KEY") or os.environ.get("CSCS_INFERENCE_API_KEY")):
    from getpass import getpass
    os.environ["LLM_API_KEY"] = getpass("Inference API key (hidden): ").strip()


In [12]:
def run_docker(request, task):
    if not LIVE_RUN:
        print("Live inference disabled. Set LIVE_RUN=True to use API credit.")
        return None
    key = (os.environ.get("LLM_API_KEY") or os.environ.get("CSCS_INFERENCE_API_KEY") or "").strip()
    if not key:
        raise RuntimeError("Set LLM_API_KEY privately before running.")
    if not (PROJECT / "track_2a" / "Makefile").is_file():
        raise RuntimeError("Update PROJECT to the repository root.")
    run_id = uuid.uuid4().hex
    folder = PROJECT / "track_2a" / "data" / "notebook-demo" / run_id
    folder.mkdir(parents=True)
    (folder / "request.json").write_text(json.dumps(request, ensure_ascii=False), encoding="utf-8")
    env = os.environ.copy()
    env["LLM_API_KEY"] = key
    env.update(LLM_NAME=MODEL, LLM_BASE_URL=BASE_URL, LLM_TOKEN_PREFLIGHT="optional",
               REQUEST_FILE=f"/app/data/notebook-demo/{run_id}/request.json",
               RESULT_FILE=f"/app/data/notebook-demo/{run_id}/result.json",
               DIAGNOSTICS_FILE=f"/app/data/notebook-demo/{run_id}/diagnostics.json")
    print(f"Running repository-root make run for Task {task}...")
    process = subprocess.run(["make", "run"], cwd=PROJECT, env=env,
                             capture_output=True, text=True)
    # Do not display raw environment or diagnostics; they are unnecessary for the demo.
    if process.returncode:
        key = env["LLM_API_KEY"]
        log = (process.stdout + process.stderr).replace(key, "[REDACTED]")
        (folder / "failure.log").write_text(log, encoding="utf-8")
        raise RuntimeError(f"make run failed (exit {process.returncode}); inspect {folder / 'failure.log'} privately.")
    result = json.loads((folder / "result.json").read_text(encoding="utf-8"))
    assert set(result) == {"id", "label", "label_name", "evidence", "metrics"}
    assert result["id"] == request["id"]
    assert type(result["label"]) is int and result["label"] in (0,1,2)
    assert all(type(v) is int and v >= 0 for v in result["metrics"].values())
    (folder / "source-fingerprints.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")
    print("make run completed with exit code 0.")
    show_result(result, f"Live Task {task}")
    return result


## Execute both tasks

Successful output must include request identity, a valid numeric label, source evidence and API token/time metrics. Exit status zero verifies that the application accepted its own result. It is not independent proof of semantic evidence correctness. The observed labels are compared with the expected label below.


In [13]:
RESULTS = {}
if LIVE_RUN:
    for task, request in [("B", REQUEST_B), ("A", REQUEST_A)]:
        RESULTS[task] = run_docker(request, task)
else:
    print("Not executed: enable LIVE_RUN to generate current-version outputs.")


Running repository-root make run for Task B...
make run completed with exit code 0.


Metric,Value
input_tokens,519
output_tokens,71
inference_time_ms,1096


Running repository-root make run for Task A...
make run completed with exit code 0.


Metric,Value
input_tokens,1345
output_tokens,93
inference_time_ms,1524


In [14]:
if RESULTS:
    rows = ""
    for task, result in RESULTS.items():
        metrics = result["metrics"]
        pages = ", ".join(str(e["page"]) for e in result["evidence"] if e["page"] is not None) or "Reference / no PDF page"
        values = [task, result["label_name"], result["label"] == 0, pages,
                  metrics["input_tokens"], metrics["output_tokens"], metrics["inference_time_ms"]]
        rows += "<tr>" + "".join("<td>" + escape(str(v)) + "</td>" for v in values) + "</tr>"
    headers = ["Task", "Prediction", "Matches expected", "Evidence page", "Input tokens", "Output tokens", "Inference ms"]
    display(HTML("<h3>Current execution results</h3><table><tr>" + "".join("<th>"+h+"</th>" for h in headers) + "</tr>" + rows + "</table>"))
    print("Model:", MODEL, "| Pipeline: vote-aware-v1")
else:
    print("No current results available. No benchmark or execution success is claimed.")


Task,Prediction,Matches expected,Evidence page,Input tokens,Output tokens,Inference ms
B,entailment,True,Reference / no PDF page,519,71,1096
A,entailment,True,19,1345,93,1524


Model: swiss-ai/Apertus-v1.5-70B | Pipeline: vote-aware-v1


## Interpretation and limitations

- These are two smoke examples, not an accuracy benchmark or proof of cross-language generalisation.
- Evidence IDs and text checks establish provenance within supplied passages; reviewers should also inspect whether the evidence supports the prediction.
- Retrieval can omit decisive information. Fixed-size passages may split sentences; scanned PDFs need OCR, which is not implemented.
- The current CLI accepts one JSON request at a time. This notebook demonstrates two separate `make run` calls and does not establish mixed-JSONL batch compliance.
- Inference time is the API completion latency, excluding Docker startup, PDF extraction, retrieval and initial downloads.
- Source fingerprints identify the local application files. They do not attest the remote model’s underlying weights or establish byte-for-byte cross-platform reproducibility.

For architecture, development evaluation and recorded experimental limitations, consult `track_2a/technical_report.md` and `track_2a/docs/P0_CORRECTNESS.md`. Historical experiments used different pipelines; their results should not be attributed to these notebook runs.


